# Reusable template — all-categorical binary classification

**Short name (GitHub):** `MushEdib` template.

Swap the CSV path, the target column, the missing token, and the costly class. Trees first (LabelEncoder is enough). Add one-hot only if you also fit a linear / distance model.

Not domain advice. The mushroom project is one instance of this skeleton.



## Knobs


In [ ]:
DATA_PATH = "data/mushrooms.csv"
TARGET = "class"
POSITIVE = "p"          # the class whose miss is costly (poisonous / default / defect)
MISSING_TOKEN = "?"
MISSING_FILL = "u"      # or "missing"
DROP_CONST = True
TEST_SIZE = 0.20
SEED = 42
N_ESTIMATORS = 100
TOP_K_CORR = 12
TOP_K_IMP = 5


## Load, clean, EDA, encode, split, fit, evaluate, simulate


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, recall_score

sns.set_theme(style="whitegrid")
df = pd.read_csv(DATA_PATH)
print(df.shape, df[TARGET].value_counts().to_dict())

# missing token → fill (scan all object columns)
obj = df.select_dtypes(include="object").columns
for c in obj:
    n = int((df[c] == MISSING_TOKEN).sum())
    if n:
        print(f"{c}: {n} '{MISSING_TOKEN}' → {MISSING_FILL}")
        df[c] = df[c].replace(MISSING_TOKEN, MISSING_FILL)

n_dups = int(df.duplicated().sum())
print("dups", n_dups)
if n_dups:
    df = df.drop_duplicates().reset_index(drop=True)

if DROP_CONST:
    const = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print("drop constant", const)
    df = df.drop(columns=const)

sns.countplot(data=df, x=TARGET); plt.title("class balance"); plt.show()

# optional: factorize heatmap
enc = df.copy()
for c in enc.columns:
    enc[c], _ = pd.factorize(enc[c])
corr = enc.corr()[TARGET].abs().drop(TARGET).sort_values(ascending=False).head(TOP_K_CORR)
print(corr)
sns.heatmap(enc[list(corr.index)].corr(), cmap="Purples", annot=False)
plt.title(f"top {TOP_K_CORR} factorize |corr|"); plt.show()

# LabelEncode + split + RF
work = df.copy()
encoders = {}
for c in work.columns:
    le = LabelEncoder()
    work[c] = le.fit_transform(work[c].astype(str))
    encoders[c] = le
X = work.drop(columns=[TARGET]); y = work[TARGET]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
clf = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=SEED)
clf.fit(Xtr, ytr)
yp = clf.predict(Xte)
print("acc", accuracy_score(yte, yp))
print(confusion_matrix(yte, yp))
print(classification_report(yte, yp))
imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(TOP_K_IMP))
imp.head(TOP_K_IMP).plot(kind="barh"); plt.title("top importances"); plt.show()

# simulation: depth
for d in [1, 2, 4, 8, None]:
    m = RandomForestClassifier(n_estimators=50, max_depth=d, random_state=SEED)
    m.fit(Xtr, ytr)
    print("depth", d, "acc", round(accuracy_score(yte, m.predict(Xte)), 4))

